##### What inferSchema=True actually does

When you set inferSchema=True, Spark performs a full extra scan of your data before loading it — just to figure out the data types. This means your CSV is read twice:


In [0]:
# This triggers TWO full reads of the file:
df = spark.read \
    .option("inferSchema", "true") \
    .option("header", "true") \
    .csv("s3://bucket/large-file.csv")

# Pass 1 → reads entire file to infer types
# Pass 2 → reads entire file again to load the data

 With inferSchema=True, Spark does a full extra pass over the data to determine types before loading. For large files this doubles read I/O. Best practice: define the schema explicitly using StructType to avoid the extra scan and prevent type surprises.

For a small file this is barely noticeable. For a multi-GB CSV on cloud storage, it doubles your I/O cost, doubles your read time, and doubles your egress charges.

##### The four real risks

**Risk 1 — Double I/O cost**

The extra scan is not a metadata operation — Spark reads every byte of the file to sample values and determine types. There's no shortcut.

**Risk 2 — Wrong type inference**

Spark samples values and makes a best guess. It can and does get it wrong:

In [0]:
# Your CSV column "order_id" contains:
# 1001, 1002, 1003, ..., 9999, 10001A  ← last value has a letter

# inferSchema sees mostly integers, infers LongType
# then fails or corrupts data when it hits "10001A"

# Or worse — your "phone_number" column:
# 0712345678  ← leading zero
# inferSchema infers LongType → 712345678  ← leading zero silently dropped!

**Risk 3 — Non-deterministic schemas**

If you run the same read on different days as new data arrives with different values, Spark may infer a different schema each time. A column that was IntegerType yesterday becomes LongType today when a large value appears. This breaks downstream code silently.

**Risk 4 — Slow job startup in pipelines**

In a production pipeline that runs every hour, paying the double-read penalty on every single run adds up significantly — especially when the schema never actually changes.

![image_1777225630741.png](./image_1777225630741.png "image_1777225630741.png")

##### Nested StructType — Defining complex hierarchical schemas

A nested StructType allows you to define schemas with complex, hierarchical structures — like JSON objects containing other objects or arrays. This is essential when working with semi-structured data.

**Why use nested StructType?**
* Represents complex real-world data structures (e.g., orders with line items, users with addresses)
* Ensures type safety at multiple levels of nesting
* Avoids the double-read penalty of inferSchema on complex data
* Makes your schema explicit and self-documenting

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, DoubleType

# Example 1: Simple nested structure (address within a person)
person_schema = StructType([
    StructField("name", StringType(), nullable=False),
    StructField("age", IntegerType(), nullable=True),
    StructField("address", StructType([
        StructField("street", StringType(), nullable=True),
        StructField("city", StringType(), nullable=True),
        StructField("zipcode", StringType(), nullable=True)
    ]), nullable=True)
])

# Example 2: Array of structs (order with multiple line items)
order_schema = StructType([
    StructField("order_id", StringType(), nullable=False),
    StructField("customer_name", StringType(), nullable=False),
    StructField("items", ArrayType(
        StructType([
            StructField("product_id", StringType(), nullable=False),
            StructField("quantity", IntegerType(), nullable=False),
            StructField("price", DoubleType(), nullable=False)
        ])
    ), nullable=True)
])

# Example 3: Deep nesting (company → departments → employees)
company_schema = StructType([
    StructField("company_name", StringType(), nullable=False),
    StructField("departments", ArrayType(
        StructType([
            StructField("dept_name", StringType(), nullable=False),
            StructField("employees", ArrayType(
                StructType([
                    StructField("emp_id", StringType(), nullable=False),
                    StructField("name", StringType(), nullable=False),
                    StructField("salary", DoubleType(), nullable=True)
                ])
            ), nullable=True)
        ])
    ), nullable=True)
])

print("Schemas defined successfully!")

In [0]:
# Using the nested schema to read JSON data
# df = spark.read.schema(person_schema).json("path/to/data.json")

# Or create sample data to demonstrate
sample_data = [
    {
        "name": "Alice",
        "age": 30,
        "address": {
            "street": "123 Main St",
            "city": "Seattle",
            "zipcode": "98101"
        }
    },
    {
        "name": "Bob",
        "age": 25,
        "address": {
            "street": "456 Oak Ave",
            "city": "Portland",
            "zipcode": "97201"
        }
    }
]

df = spark.createDataFrame(sample_data, schema=person_schema)

print("DataFrame Schema:")
df.printSchema()

print("\nSample Data:")
display(df)

##### Accessing nested fields

Once you have a DataFrame with nested structures, you can access nested fields using dot notation or bracket notation:

In [0]:
# Method 1: Dot notation (preferred)
df.select("name", "address.city", "address.zipcode").show()

# Method 2: Using col() function
from pyspark.sql.functions import col
df.select(
    col("name"),
    col("address.city").alias("city"),
    col("address.street").alias("street")
).show()

# Filtering on nested fields
df.filter(col("address.city") == "Seattle").show()

##### Advanced Nested Structures — Real-world complexity

Beyond simple nesting, production data often involves:
* **Deep nesting** (4+ levels) — API responses, configuration files
* **Arrays of arrays** — time-series data with multiple metrics
* **Maps within structs** — dynamic key-value metadata
* **Mixed complexity** — combining all of the above

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    DoubleType, ArrayType, MapType, TimestampType, BooleanType
)

# Real-world e-commerce order schema (4 levels deep)
ecommerce_schema = StructType([
    StructField("order_id", StringType(), nullable=False),
    StructField("order_date", TimestampType(), nullable=False),
    StructField("customer", StructType([
        StructField("customer_id", StringType(), nullable=False),
        StructField("name", StringType(), nullable=False),
        StructField("email", StringType(), nullable=True),
        StructField("addresses", ArrayType(
            StructType([
                StructField("type", StringType(), nullable=False),  # billing, shipping
                StructField("street", StringType(), nullable=False),
                StructField("city", StringType(), nullable=False),
                StructField("state", StringType(), nullable=False),
                StructField("zipcode", StringType(), nullable=False),
                StructField("coordinates", StructType([  # Level 4 nesting!
                    StructField("latitude", DoubleType(), nullable=True),
                    StructField("longitude", DoubleType(), nullable=True)
                ]), nullable=True)
            ])
        ), nullable=True),
        StructField("loyalty_tier", StringType(), nullable=True)
    ]), nullable=False),
    StructField("items", ArrayType(
        StructType([
            StructField("sku", StringType(), nullable=False),
            StructField("product_name", StringType(), nullable=False),
            StructField("quantity", IntegerType(), nullable=False),
            StructField("unit_price", DoubleType(), nullable=False),
            StructField("discounts", ArrayType(  # Nested array!
                StructType([
                    StructField("code", StringType(), nullable=False),
                    StructField("amount", DoubleType(), nullable=False),
                    StructField("type", StringType(), nullable=False)  # percentage, fixed
                ])
            ), nullable=True),
            StructField("metadata", MapType(StringType(), StringType()), nullable=True)  # Dynamic attributes
        ])
    ), nullable=False),
    StructField("payment", StructType([
        StructField("method", StringType(), nullable=False),
        StructField("transaction_id", StringType(), nullable=False),
        StructField("amount", DoubleType(), nullable=False),
        StructField("installments", IntegerType(), nullable=True)
    ]), nullable=False),
    StructField("tags", MapType(StringType(), StringType()), nullable=True)  # Custom metadata
])

print("E-commerce schema created with 4 levels of nesting")
print(f"Schema depth: customer → addresses → coordinates (4 levels)")
print(f"Array of arrays: items → discounts")
print(f"Map fields: metadata, tags")

In [0]:
# Event tracking system with nested time-series metrics
event_log_schema = StructType([
    StructField("event_id", StringType(), nullable=False),
    StructField("timestamp", TimestampType(), nullable=False),
    StructField("event_type", StringType(), nullable=False),
    StructField("user_session", StructType([
        StructField("session_id", StringType(), nullable=False),
        StructField("user_id", StringType(), nullable=False),
        StructField("device_info", StructType([
            StructField("device_type", StringType(), nullable=True),
            StructField("os", StringType(), nullable=True),
            StructField("browser", StringType(), nullable=True),
            StructField("screen_resolution", StructType([
                StructField("width", IntegerType(), nullable=True),
                StructField("height", IntegerType(), nullable=True)
            ]), nullable=True)
        ]), nullable=True),
        StructField("user_actions", ArrayType(  # Array of action sequences
            StructType([
                StructField("action_type", StringType(), nullable=False),
                StructField("timestamp", TimestampType(), nullable=False),
                StructField("properties", MapType(StringType(), StringType()), nullable=True)
            ])
        ), nullable=True)
    ]), nullable=True),
    StructField("performance_metrics", StructType([
        StructField("page_load_time_ms", IntegerType(), nullable=True),
        StructField("api_calls", ArrayType(  # Array of API call measurements
            StructType([
                StructField("endpoint", StringType(), nullable=False),
                StructField("duration_ms", IntegerType(), nullable=False),
                StructField("status_code", IntegerType(), nullable=False),
                StructField("response_size_bytes", IntegerType(), nullable=True)
            ])
        ), nullable=True),
        StructField("resource_timings", ArrayType(  # Nested performance data
            StructType([
                StructField("resource_url", StringType(), nullable=False),
                StructField("timings", StructType([  # Deep nesting for timing breakdown
                    StructField("dns_lookup_ms", IntegerType(), nullable=True),
                    StructField("tcp_connection_ms", IntegerType(), nullable=True),
                    StructField("request_ms", IntegerType(), nullable=True),
                    StructField("response_ms", IntegerType(), nullable=True)
                ]), nullable=True)
            ])
        ), nullable=True)
    ]), nullable=True)
])

print("Event log schema created with complex nested arrays and maps")
print("Features: nested time-series, performance metrics, dynamic properties")

In [0]:
from pyspark.sql.functions import col, explode, explode_outer, size, map_keys, map_values
from datetime import datetime

# Sample complex data
complex_data = [{
    "order_id": "ORD-001",
    "order_date": datetime.fromisoformat("2026-04-26T10:30:00"),
    "customer": {
        "customer_id": "CUST-123",
        "name": "Jane Doe",
        "email": "jane@example.com",
        "addresses": [
            {
                "type": "shipping",
                "street": "123 Main St",
                "city": "Seattle",
                "state": "WA",
                "zipcode": "98101",
                "coordinates": {"latitude": 47.6062, "longitude": -122.3321}
            },
            {
                "type": "billing",
                "street": "456 Oak Ave",
                "city": "Portland",
                "state": "OR",
                "zipcode": "97201",
                "coordinates": {"latitude": 45.5152, "longitude": -122.6784}
            }
        ],
        "loyalty_tier": "Gold"
    },
    "items": [
        {
            "sku": "LAPTOP-001",
            "product_name": "Ultra Laptop Pro",
            "quantity": 1,
            "unit_price": 1299.99,
            "discounts": [
                {"code": "SAVE10", "amount": 130.0, "type": "percentage"},
                {"code": "WELCOME", "amount": 50.0, "type": "fixed"}
            ],
            "metadata": {"color": "silver", "warranty": "2-year"}
        },
        {
            "sku": "MOUSE-042",
            "product_name": "Wireless Mouse",
            "quantity": 2,
            "unit_price": 29.99,
            "discounts": None,
            "metadata": {"color": "black"}
        }
    ],
    "payment": {
        "method": "credit_card",
        "transaction_id": "TXN-789456",
        "amount": 1179.97,
        "installments": 3
    },
    "tags": {"source": "mobile_app", "campaign": "spring_sale"}
}]

df = spark.createDataFrame(complex_data, schema=ecommerce_schema)

print("Original nested structure:")
df.printSchema()
print("\nSample data:")
display(df)

In [0]:
# Technique 1: Direct dot notation (up to 4 levels deep)
flattened = df.select(
    col("order_id"),
    col("customer.name").alias("customer_name"),
    col("customer.loyalty_tier").alias("tier"),
    col("customer.addresses"),  # Keep array as-is
    col("payment.method").alias("payment_method"),
    col("payment.amount").alias("total_amount")
)

print("Accessing nested fields with dot notation:")
display(flattened)

In [0]:
# Technique 2: Exploding arrays to flatten
# Explode items array
items_exploded = df.select(
    col("order_id"),
    col("customer.name").alias("customer_name"),
    explode("items").alias("item")
)

print("After exploding items array:")
display(items_exploded)

# Further explode discounts within each item
from pyspark.sql.functions import coalesce, lit

items_with_discounts = items_exploded.select(
    col("order_id"),
    col("customer_name"),
    col("item.sku"),
    col("item.product_name"),
    col("item.unit_price"),
    explode_outer("item.discounts").alias("discount")  # explode_outer keeps rows with null discounts
)

print("\nAfter exploding discounts array (nested explosion):")
display(items_with_discounts)

In [0]:
# Technique 3: Working with MapType fields
# Extract specific map keys
with_metadata = df.select(
    col("order_id"),
    explode("items").alias("item")
).select(
    col("order_id"),
    col("item.sku"),
    col("item.metadata"),
    col("item.metadata.color").alias("product_color"),  # Access specific key
    col("item.metadata.warranty").alias("warranty"),
    map_keys("item.metadata").alias("available_attributes"),  # Get all keys
    map_values("item.metadata").alias("attribute_values")  # Get all values
)

print("Working with Map fields:")
display(with_metadata)

In [0]:
# Technique 4: Accessing 4-level deep nesting
# customer → addresses → coordinates → latitude/longitude

addresses_with_coords = df.select(
    col("order_id"),
    col("customer.name").alias("customer_name"),
    explode("customer.addresses").alias("address")
).select(
    col("order_id"),
    col("customer_name"),
    col("address.type").alias("address_type"),
    col("address.city"),
    col("address.state"),
    col("address.coordinates.latitude").alias("lat"),  # 4 levels deep!
    col("address.coordinates.longitude").alias("lon")
)

print("Accessing 4-level deep nested coordinates:")
display(addresses_with_coords)

##### Best Practices for Complex Nested Structures

**Schema Design:**
* Define schemas explicitly — don't rely on `inferSchema` for complex nesting
* Use meaningful field names at every level
* Document nullable fields carefully (especially in arrays)
* Consider flattening frequently-accessed deep fields

**Performance:**
* Avoid repeated `explode()` on large arrays — flatten once and reuse
* Use `explode_outer()` to preserve rows with null/empty arrays
* Cache intermediate results after expensive explosions
* Consider partitioning by top-level keys before exploding

**Query Patterns:**
* Dot notation for direct access: `col("level1.level2.level3")`
* `explode()` to convert arrays to rows
* `collect_list()` to reaggregate after exploding
* `struct()` to create new nested structures
* `map_keys()`, `map_values()` for MapType fields